# COMP64702 RAG Coursework
## Notebook 2: RAG Pipeline

This notebook implements the full Retrieval-Augmented Generation (RAG) pipeline:
- **Chunking** – section-aware splitting with sentence-level fallback for long documents
- **Vectorisation / Embedding** – `all-MiniLM-L6-v2` dense embeddings
- **Retrieval & Ranking** – Hybrid BM25 + dense retrieval with Reciprocal Rank Fusion (RRF)
- **Prompting & Generation** – Qwen2.5-0.5B-Instruct with structured prompt
- **Output** – saves `test_outputs.json` in the required format

**Files used:**
- `Background_Corpus_Final_Combined.json` – 200-entry South Asian cuisine corpus
- `benchmark_input_only.json` – 100 queries (no gold answers)
- `test_outputs.json` – generated answers (output)

In [1]:
# Install dependencies (run once)
!pip install sentence-transformers transformers rank-bm25 accelerate -q

In [2]:
# Imports
import json
import re
import math
import numpy as np
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from rank_bm25 import BM25Okapi
import torch

print('All imports OK')

/opt/miniconda3/envs/rag-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports OK


In [3]:
# File paths
DATA_DIR = Path('.')

corpus_path = DATA_DIR / 'background_corpus.json'
query_path  = DATA_DIR / 'benchmark_input_only.json'
output_path = DATA_DIR / 'test_outputs.json'

with open(corpus_path, 'r', encoding='utf-8') as f:
    corpus = json.load(f)

with open(query_path, 'r', encoding='utf-8') as f:
    query_payload = json.load(f)

queries = query_payload['queries']
print(f'Corpus: {len(corpus)} documents')
print(f'Queries: {len(queries)}')

Corpus: 200 documents
Queries: 100


In [4]:
# Data structures

@dataclass
class Chunk:
    doc_id:  str
    title:   str
    section: str
    text:    str

@dataclass
class RetrievalResult:
    chunk: Chunk
    score: float

In [5]:
# ── Chunking ──────────────────────────────────────────────────────────────
# Strategy:
#   1. If a document has named sections, each section becomes one chunk.
#   2. If a document has only full_text, split into ~400-word sentence-boundary
#      chunks with a 50-word overlap so context is not lost at boundaries.
# This produces richer, more retrievable units than treating 1700-char entries
# as single monolithic chunks.

CHUNK_WORDS   = 400   # target chunk size in words
OVERLAP_WORDS = 50    # overlap between adjacent chunks

def split_into_chunks(text: str, max_words=CHUNK_WORDS, overlap=OVERLAP_WORDS) -> List[str]:
    """Split text into overlapping word-boundary chunks."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks, current, current_len = [], [], 0
    for sent in sentences:
        words = sent.split()
        if current_len + len(words) > max_words and current:
            chunks.append(' '.join(current))
            # keep overlap
            overlap_sents = []
            overlap_len = 0
            for s in reversed(current):
                wlen = len(s.split())
                if overlap_len + wlen > overlap:
                    break
                overlap_sents.insert(0, s)
                overlap_len += wlen
            current = overlap_sents
            current_len = overlap_len
        current.append(sent)
        current_len += len(words)
    if current:
        chunks.append(' '.join(current))
    return chunks


def chunk_documents(corpus_docs: list) -> List[Chunk]:
    chunks = []
    for doc in corpus_docs:
        doc_id  = doc.get('doc_id', '')
        title   = doc.get('title', '')
        sections = doc.get('sections', [])

        # Skip meta/scaffold entries — they add noise
        if 'scaffold' in doc_id or 'scope' in doc_id:
            continue

        if sections:
            # Section-level chunks
            for sec in sections:
                heading = sec.get('heading', 'Section')
                text = ' '.join(sec.get('content', [])).strip()
                if len(text.split()) >= 20:
                    chunks.append(Chunk(doc_id=doc_id, title=title,
                                        section=heading, text=text))
        else:
            full_text = doc.get('full_text', '').strip()
            if not full_text:
                continue
            word_count = len(full_text.split())
            if word_count <= CHUNK_WORDS:
                # Short enough: single chunk
                chunks.append(Chunk(doc_id=doc_id, title=title,
                                    section='Full text', text=full_text))
            else:
                # Long: split with overlap
                for i, part in enumerate(split_into_chunks(full_text)):
                    chunks.append(Chunk(doc_id=doc_id, title=title,
                                        section=f'Part {i+1}', text=part))
    return chunks


chunks = chunk_documents(corpus)
print(f'Total chunks: {len(chunks)}')
print(f'Avg chunk length (words): {sum(len(c.text.split()) for c in chunks) // len(chunks)}')

# Show distribution
from collections import Counter
multi = Counter(c.doc_id for c in chunks)
print(f'Docs with >1 chunk: {sum(1 for v in multi.values() if v > 1)}')

Total chunks: 276
Avg chunk length (words): 191
Docs with >1 chunk: 13


In [6]:
# ── Embedding ─────────────────────────────────────────────────────────────
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print('Model loaded:', embedding_model.get_sentence_embedding_dimension(), 'dims')

# Prefix each chunk text with its title for richer embeddings
chunk_texts = [f"{c.title}: [{c.section}] {c.text}" for c in chunks]

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    batch_size=64,
    show_progress_bar=True
)

# Pre-normalise for fast cosine similarity via dot product
norms = np.linalg.norm(chunk_embeddings, axis=1, keepdims=True) + 1e-12
chunk_embeddings_norm = chunk_embeddings / norms

print(f'Embeddings shape: {chunk_embeddings_norm.shape}')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9466.09it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded: 384 dims


Batches: 100%|██████████| 5/5 [00:01<00:00,  2.89it/s]

Embeddings shape: (276, 384)


In [7]:
# ── BM25 ───────────────────────────────────────────────────────────────────
def tokenize(text: str) -> List[str]:
    """Simple whitespace + lowercase tokeniser for BM25."""
    return re.findall(r'\b[a-z0-9]+\b', text.lower())

tokenized_chunks = [tokenize(t) for t in chunk_texts]
bm25 = BM25Okapi(tokenized_chunks)
print('BM25 index built over', len(tokenized_chunks), 'chunks')

BM25 index built over 276 chunks


In [8]:
# ── Dense retrieval ───────────────────────────────────────────────────────
def dense_retrieve(query: str, top_k: int = 10) -> List[RetrievalResult]:
    q_emb = embedding_model.encode([query], convert_to_numpy=True)[0]
    q_emb = q_emb / (np.linalg.norm(q_emb) + 1e-12)
    sims = chunk_embeddings_norm @ q_emb
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [RetrievalResult(chunk=chunks[i], score=float(sims[i])) for i in top_idx]

In [9]:
# ── BM25 retrieval ────────────────────────────────────────────────────────
def bm25_retrieve(query: str, top_k: int = 10) -> List[RetrievalResult]:
    scores = bm25.get_scores(tokenize(query))
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [RetrievalResult(chunk=chunks[i], score=float(scores[i])) for i in top_idx]

In [10]:
# ── Hybrid retrieval (RRF) ────────────────────────────────────────────────
# Reciprocal Rank Fusion combines dense and BM25 ranked lists.
# RRF_K=60 is the standard constant (Cormack et al., 2009).
# Diversification: max 2 chunks per doc_id to avoid one document
# dominating the context window.

RRF_K    = 60
TOP_K    = 5
MAX_PER_DOC = 2

def hybrid_retrieve(query: str, top_k: int = TOP_K) -> List[RetrievalResult]:
    dense_res = dense_retrieve(query, top_k=10)
    bm25_res  = bm25_retrieve(query,  top_k=10)

    # RRF scoring
    score_map: Dict[str, dict] = {}
    for rank, res in enumerate(dense_res, 1):
        key = f"{res.chunk.doc_id}||{res.chunk.section}||{res.chunk.text[:80]}"
        score_map.setdefault(key, {'chunk': res.chunk, 'score': 0.0})
        score_map[key]['score'] += 1.0 / (RRF_K + rank)

    for rank, res in enumerate(bm25_res, 1):
        key = f"{res.chunk.doc_id}||{res.chunk.section}||{res.chunk.text[:80]}"
        score_map.setdefault(key, {'chunk': res.chunk, 'score': 0.0})
        score_map[key]['score'] += 1.0 / (RRF_K + rank)

    merged = sorted(score_map.values(), key=lambda x: x['score'], reverse=True)

    # Diversify: max MAX_PER_DOC chunks from same document
    selected, doc_counts = [], {}
    for item in merged:
        did = item['chunk'].doc_id
        if doc_counts.get(did, 0) >= MAX_PER_DOC:
            continue
        selected.append(RetrievalResult(chunk=item['chunk'], score=item['score']))
        doc_counts[did] = doc_counts.get(did, 0) + 1
        if len(selected) == top_k:
            break

    return selected


# ── Baseline: dense-only (for comparison) ──────────────────────────────────
def baseline_retrieve(query: str, top_k: int = TOP_K) -> List[RetrievalResult]:
    """Dense-only retrieval — used as baseline in evaluation."""
    return dense_retrieve(query, top_k=top_k)

In [11]:
# ── Smoke test retrieval ──────────────────────────────────────────────────
test_query = 'What is biryani and which region is it originally associated with?'
test_results = hybrid_retrieve(test_query)

print(f'Top {len(test_results)} chunks retrieved for: "{test_query}"\n')
for i, r in enumerate(test_results, 1):
    print(f'{i}. [{r.chunk.doc_id} | {r.chunk.section}]  score={r.score:.4f}')
    print(f'   {r.chunk.text[:120]}...')
    print()

Top 5 chunks retrieved for: "What is biryani and which region is it originally associated with?"

1. [wiki_biryani | Introduction]  score=0.0325
   Biryani is a rice dish made with rice, spices, and usually meat, although vegetarian versions also exist. It is one of t...

2. [bangladesh_dish_kacchi_biryani | Full text]  score=0.0306
   Kacchi biryani (Bengali: কাচ্চি বিরিয়ানি, meaning "raw biryani") is the most celebrated and prestigious biryani traditi...

3. [pakistan_dish_biryani | Full text]  score=0.0299
   Pakistani biryani is a rich, aromatic rice dish made by layering partially cooked basmati rice with a spiced meat or veg...

4. [india_regional_hyderabadi_cuisine | Full text]  score=0.0294
   Hyderabadi cuisine (also called Deccani cuisine) is the culinary tradition of the city of Hyderabad and its historic sur...

5. [wiki_biryani | Regional forms]  score=0.0164
   There are many regional varieties of biryani across the Indian subcontinent. The dish is popular in everyday di

In [12]:
# ── Load Qwen2.5-0.5B-Instruct ───────────────────────────────────────────
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype='auto',
    device_map='auto'
)
model.eval()
print(f'Model loaded on: {next(model.parameters()).device}')

Loading weights: 100%|██████████| 290/290 [00:01<00:00, 236.84it/s]


Model loaded on: mps:0


In [13]:
# ── Prompt builder ────────────────────────────────────────────────────────
# Improvements over v1:
#   - System prompt explicitly forbids fabrication
#   - Context includes doc title for better attribution
#   - Instruction tailored per question type (comparative vs procedural)
#   - Max context truncated to 1800 chars per chunk to avoid token overflow

MAX_CONTEXT_CHARS = 1800

def build_prompt(query: str, retrieved: List[RetrievalResult]) -> List[dict]:
    context_parts = []
    for r in retrieved:
        text = r.chunk.text[:MAX_CONTEXT_CHARS]
        context_parts.append(
            f"[Source: {r.chunk.title} | {r.chunk.section}]\n{text}"
        )
    context = '\n\n---\n\n'.join(context_parts)

    # Detect question intent for targeted instruction
    q_lower = query.lower()
    if any(w in q_lower for w in ['differ', 'difference', 'distinguish', 'compare', 'vs', 'versus']):
        task_hint = 'This is a comparison question. Identify the key differences supported by the sources.'
    elif any(w in q_lower for w in ['how is', 'how are', 'how do', 'prepared', 'made', 'cook']):
        task_hint = 'This is a procedural question. Describe the main steps or method supported by the sources.'
    elif any(w in q_lower for w in ['what is', 'what are', 'define', 'describe']):
        task_hint = 'This is a factual question. Provide a clear, direct answer supported by the sources.'
    else:
        task_hint = 'Answer the question using only information from the sources below.'

    return [
        {
            'role': 'system',
            'content': (
                'You are a knowledgeable South Asian cuisine assistant. '
                'Answer questions using ONLY the information in the retrieved sources. '
                'Do NOT add facts not present in the sources. '
                'Be concise and accurate. '
                'Do not mention "retrieved sources" or "context" in your answer.'
            )
        },
        {
            'role': 'user',
            'content': (
                f'Retrieved sources:\n{context}\n\n'
                f'Question: {query}\n\n'
                f'{task_hint} '
                f'Write your answer in 2 to 3 sentences.'
            )
        }
    ]

In [14]:
# ── Generation ────────────────────────────────────────────────────────────
def postprocess(text: str) -> str:
    text = text.strip()
    # Remove common artefacts
    for prefix in ['Answer:', 'Response:', 'A:']:
        if text.startswith(prefix):
            text = text[len(prefix):].strip()
    return re.sub(r'\s+', ' ', text).strip()


def generate(messages: list) -> str:
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([prompt_text], return_tensors='pt').to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=120,       # increased from 70 for fuller answers
            do_sample=False,
            temperature=None,
            top_p=None,
            repetition_penalty=1.1,   # slightly stronger to reduce repetition
            pad_token_id=tokenizer.eos_token_id
        )

    new_ids = output_ids[0][inputs['input_ids'].shape[-1]:]
    return postprocess(tokenizer.decode(new_ids, skip_special_tokens=True))

In [15]:
# ── Single query test ─────────────────────────────────────────────────────
sample_query   = queries[0]['query']
sample_results = hybrid_retrieve(sample_query)
sample_messages = build_prompt(sample_query, sample_results)
sample_answer  = generate(sample_messages)

print('QUERY:  ', sample_query)
print('ANSWER: ', sample_answer)
print()
print('Retrieved docs:')
for r in sample_results:
    print(f'  - {r.chunk.doc_id} | {r.chunk.section}')

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


QUERY:   What is biryani and which culinary tradition is it most closely associated with?
ANSWER:  Biryani is a rice dish made with rice, spices, and often meat, typically associated with South Asian cuisine. It is most closely associated with the Pakistani cuisine tradition, particularly the Sindhi, Lahori, and Karachi styles.

Retrieved docs:
  - wiki_biryani | Introduction
  - bangladesh_dish_kacchi_biryani | Full text
  - india_regional_hyderabadi_cuisine | Full text
  - pakistan_dish_biryani | Full text
  - india_dish_biryani | Full text


In [16]:
# ── Full inference loop ───────────────────────────────────────────────────
results = []

for i, item in enumerate(queries):
    query_id = item['query_id']
    query    = item['query']

    retrieved = hybrid_retrieve(query, top_k=TOP_K)
    messages  = build_prompt(query, retrieved)
    response  = generate(messages)

    results.append({
        'query_id': query_id,
        'query':    query,
        'response': response,
        'retrieved_context': [
            {
                'doc_id': r.chunk.doc_id,
                'text':   f'[{r.chunk.section}] {r.chunk.text[:500]}'
            }
            for r in retrieved
        ]
    })

    if (i + 1) % 10 == 0:
        print(f'  {i+1}/{len(queries)} done')

print(f'\nGenerated {len(results)} answers')

  10/100 done
  20/100 done
  30/100 done
  40/100 done
  50/100 done
  60/100 done
  70/100 done
  80/100 done
  90/100 done
  100/100 done

Generated 100 answers


In [17]:
# ── Save test_outputs.json ────────────────────────────────────────────────
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump({'results': results}, f, indent=2, ensure_ascii=False)

print(f'Saved {len(results)} results to {output_path}')

Saved 100 results to test_outputs.json


In [18]:
# ── Baseline comparison (dense-only) ─────────────────────────────────────
# Generate answers using dense-only retrieval for comparison.
# These are saved to test_outputs_baseline.json.
# The evaluation notebook compares hybrid vs baseline scores.

baseline_results = []

for item in queries:
    query_id = item['query_id']
    query    = item['query']

    retrieved = baseline_retrieve(query, top_k=TOP_K)
    messages  = build_prompt(query, retrieved)
    response  = generate(messages)

    baseline_results.append({
        'query_id': query_id,
        'query':    query,
        'response': response,
        'retrieved_context': [
            {'doc_id': r.chunk.doc_id, 'text': f'[{r.chunk.section}] {r.chunk.text[:500]}'}
            for r in retrieved
        ]
    })

baseline_path = DATA_DIR / 'test_outputs_baseline.json'
with open(baseline_path, 'w', encoding='utf-8') as f:
    json.dump({'results': baseline_results}, f, indent=2, ensure_ascii=False)

print(f'Baseline results saved to {baseline_path}')

Baseline results saved to test_outputs_baseline.json


In [23]:
# ── Interactive demo ──────────────────────────────────────────────────────
# Run this cell during the live demonstration.
# Type any South Asian cuisine question to see retrieval + generation.

demo_query = input('Enter your question: ')

demo_retrieved = hybrid_retrieve(demo_query)
demo_messages  = build_prompt(demo_query, demo_retrieved)
demo_answer    = generate(demo_messages)

print('\n── ANSWER ─────────────────────────────────────────────────')
print(demo_answer)

print('\n── RETRIEVED CONTEXT ──────────────────────────────────────')
for i, r in enumerate(demo_retrieved, 1):
    print(f'\n{i}. [{r.chunk.doc_id} | {r.chunk.section}]  (RRF score: {r.score:.5f})')
    print(r.chunk.text[:300])


── ANSWER ─────────────────────────────────────────────────
Ghevar is typically served alongside a variety of other dishes in Rajasthani cuisine. For example, it can be paired with: 1. Dal Baati Churma - A popular dessert that combines hard wheat flour balls with a robust five-lentil dal and churma. 2. Ker Sangri - A savory desert that includes dried ker berries cooked with dried sangri beans. 3. Gatte Ki Sabzi - A traditional Rajasthani dish consisting of dried ker berries cooked with dried sangri beans. 4. Laal Maas - A spicy red goat curry made with dried mathania

── RETRIEVED CONTEXT ──────────────────────────────────────

1. [wiki_sri_lankan_cuisine | Kiribath]  (RRF score: 0.03175)
Kiribath or paal soru ( lit. 'milk rice') is rice cooked in salted coconut milk until the grains turn soft and porridge-like. Generally eaten for breakfast, kiribath is also prepared on special occasions such as birthdays, New Years' and religious festivals. It is usually served with lunu miris , a 
